# Rich-quench-lean (RQL) gas-turbine combustor

Modern low-NOx combustors burn in stages.
A **rich primary zone** (equivalence ratio $\phi \approx 1.6$) burns the fuel with too little air: it runs cooler than a stoichiometric flame and, starved of oxygen, forms little NO, but it leaves carbon monoxide and hydrogen unburnt.
**Quench air** is then injected and the mixture completes combustion in a **lean zone**, oxidizing the leftover CO and H$_2$ before the turbine inlet.

![Network topology](rql_combustor_topology.png)

The modeling subtlety this notebook exercises: the lean gas is downstream of the flame, so it must stay in chemical **equilibrium** and re-burn the CO / H$_2$ -- the quench air must *not* revert it to a cold unburnt mixture.
Nefes handles this automatically with a **sticky burnt marker**: a transported label ("am I downstream of a flame?") that a fresh stream can never dilute.
No per-edge closure is wired by hand.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.thermo import ThermoInp, EQ_FROZEN, EQ_KERNEL
from nefes.chem import species_mass_fractions, enthalpy_mass, equivalence_ratio_mixture
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()

# A compact CH4/air library selected from the NASA CEA `thermo.inp` database: the air
# species and the fuel, plus the major products the equilibrium solver fills in.
lib = ThermoInp().library(
    ["CH4", "O2", "N2", "Ar", "CO2", "H2O", "CO", "H2", "OH", "O", "H", "NO"]
)
AIR = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
h_air = enthalpy_mass(lib, species_mass_fractions(lib, AIR, "mole"), 300.0)
print("elements:", list(lib.elements))

## The combustor as a network

`rich premix -> primary duct -> equilibrium flame -> rich zone -> quench-air source -> lean zone -> turbine inlet`

Total fuel and total air are held fixed (a fixed lean overall $\phi = 0.5$); the primary equivalence ratio sets how the air is split between the rich primary zone and the quench.
The network is built with **no** explicit per-edge closure, so it runs the auto marker-gated reacting path.

In [ ]:
AREA = 0.01          # duct cross-section [m^2]
MDOT_FUEL = 0.020    # total CH4 [kg/s]  (fixed)
PHI_OVERALL = 0.50   # lean overall equivalence ratio (fixed)
# stoichiometric CH4/air mass ratio, straight from the library
_stoich = equivalence_ratio_mixture(lib, {"CH4": 1.0}, AIR, 1.0, basis="mass")
F_STOICH = _stoich["CH4"] / (1.0 - _stoich["CH4"])


def build_rql(phi_primary=1.6, mdot_fuel=MDOT_FUEL, phi_overall=PHI_OVERALL,
              T_in=450.0, p=12.0e5, edge_models=None):
    """Rich-quench-lean combustor as a Nefes network.

    Total fuel and total air are fixed (fixed lean overall phi); ``phi_primary`` sets how
    the air splits between the rich primary zone and the quench.  The primary is fed as a
    fuel + air premix; the quench air is an inline mass source downstream of the flame.
    With ``edge_models=None`` the network runs the automatic marker-gated reacting path;
    pass an explicit per-edge closure list to pin it by hand.
    """
    mdot_air_total = mdot_fuel / (F_STOICH * phi_overall)
    primary_air = mdot_fuel / (F_STOICH * phi_primary)
    quench_air = mdot_air_total - primary_air
    mdot_primary = mdot_fuel + primary_air
    rich = equivalence_ratio_mixture(lib, {"CH4": 1.0}, AIR, phi_primary)
    nodes = [
        cat.mass_flow_inlet(mdot_primary, T_in, composition=rich, basis="mole", name="rich premix"),
        cat.duct(0.25, name="primary duct"),
        cat.equilibrium_flame(name="flame"),
        cat.duct(0.25, name="rich zone"),
        cat.mass_source(quench_air, T_in, composition=AIR, basis="mole", name="quench air"),
        cat.duct(0.40, name="lean zone"),
        cat.pressure_outlet(p, T_in, composition=AIR, basis="mole", name="turbine inlet"),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA), (4, 5, AREA), (5, 6, AREA)]
    return nefes.Network(nefes.equilibrium(lib), nodes=nodes, edges=edges, edge_models=edge_models)


net = build_rql(phi_primary=1.6)
sol = net.solve()
print("converged:", sol.converged)
print(f"primary flame T = {sol.field('T')[2]:.0f} K    turbine-inlet T = {sol.field('T')[5]:.0f} K")

In [ ]:
net.plot(solution=sol, color_by="T", width_by="mdot",
         colorscale="Inferno", title="RQL combustor -- static temperature [K]")

## The sticky burnt marker

At the quench node, hot burnt products (marker $b = 1$) meet fresh air ($b = 0$).
The marker is a *reachability label*, not a conserved quantity, so it is transported by a noisy-OR: any burnt inflow keeps the outgoing gas burnt.
A naive mass-average would instead pull the marker below the $0.5$ gate crossover and wrongly freeze the lean zone -- the black cross marks where averaging would land.

In [ ]:
edge_names = ["premix", "approach", "rich-1", "rich-2", "lean-1", "lean-2"]
E = len(sol.field("mdot"))
b = np.array([sol.marker(e) for e in range(E)])
mdot = sol.field("mdot")
naive = abs(mdot[3]) / abs(mdot[4])  # burnt inflow / (burnt inflow + quench air)

fig = go.Figure()
fig.add_bar(x=edge_names, y=b, name="burnt marker b",
            marker_color=[COLORWAY[0] if bi < 0.5 else COLORWAY[4] for bi in b])
fig.add_hline(y=0.5, line_dash="dot", line_color="gray", annotation_text="gate crossover")
fig.add_scatter(x=["lean-1"], y=[naive], mode="markers+text", name="mass-average would give",
                marker=dict(size=14, symbol="x", color="black", line=dict(width=2)),
                text=[f"  averaging: {naive:.2f}"], textposition="middle right")
fig.update_layout(yaxis_title="burnt marker b", yaxis_range=[-0.05, 1.12],
                  title="Sticky marker: the quench air does not dilute the burnt label")
fig

## Axial profile

The flame jumps the rich zone to its (cooler, oxygen-limited) equilibrium, rich in CO and H$_2$.
The quench air then completes combustion: CO and H$_2$ fall to near zero, free O$_2$ appears, and the temperature settles to the turbine-inlet value.

In [ ]:
keys = ["CO", "CO2", "O2", "H2O", "H2"]
palette = {"CO": COLORWAY[4], "CO2": COLORWAY[2], "O2": COLORWAY[0], "H2O": COLORWAY[1], "H2": COLORWAY[3]}
x_ax = list(range(6))
T = sol.field("T")[:6]
prof = {k: [sol.species(e, "mole").get(k, 0.0) for e in range(6)] for k in keys}

fig = make_subplots(specs=[[{"secondary_y": True}]])
for k in keys:
    fig.add_scatter(x=x_ax, y=prof[k], name=k, mode="lines+markers", line=dict(color=palette[k]))
fig.add_scatter(x=x_ax, y=T, name="T", mode="lines+markers", secondary_y=True,
                line=dict(color="black", dash="dash", width=3))
fig.update_xaxes(tickvals=x_ax, ticktext=edge_names)
fig.update_yaxes(title_text="mole fraction", secondary_y=False)
fig.update_yaxes(title_text="static temperature [K]", secondary_y=True)
fig.update_layout(title="Axial profile: rich primary, quench, lean burnout")
fig

## Re-equilibration across the quench

The same physics, read as composition: the lean zone is *not* a frozen blend of the rich products with cold air -- it re-equilibrates, burning CO $\to$ CO$_2$ and H$_2 \to$ H$_2$O and leaving excess O$_2$.
This is the step the sticky marker makes automatic.

In [ ]:
rich_sp = sol.species(2, "mole")   # rich zone
lean_sp = sol.species(5, "mole")   # lean zone (turbine inlet)
show = ["CO2", "H2O", "CO", "H2", "O2", "OH", "NO"]
fig = go.Figure()
fig.add_bar(x=show, y=[rich_sp.get(k, 0.0) for k in show], name="rich zone", marker_color=COLORWAY[4])
fig.add_bar(x=show, y=[lean_sp.get(k, 0.0) for k in show], name="lean zone", marker_color=COLORWAY[0])
fig.update_layout(barmode="group", yaxis_title="mole fraction",
                  title="Re-equilibration across the quench: CO / H2 burn out, O2 appears")
fig

## Air-split sweep: the RQL lever

Holding the overall (lean) stoichiometry fixed and richening the primary lowers the **primary flame temperature** and collapses the **NO formed in the primary** -- the physical basis of RQL staging.
The turbine-inlet temperature is set by the fixed overall mixture, so it stays flat.

A caveat worth stating: the equilibrium closure carries no reaction-rate memory, so the *lean-exit* NO it reports is the equilibrium value at the fixed overall state, not the kinetically frozen emission.
The RQL benefit shows up here as the suppressed *primary* NO; capturing the frozen exit level is a job for the finite-rate closure (deferred).

In [ ]:
phis = np.linspace(1.05, 2.0, 11)
Tprim, Texit, NOprim = [], [], []
prev = None
for pr in phis:
    s = build_rql(phi_primary=float(pr)).solve(x0=prev.x if prev else None)
    prev = s
    Tprim.append(s.field("T")[2])
    Texit.append(s.field("T")[5])
    NOprim.append(s.species(2, "mole").get("NO", 0.0) * 1e6)  # ppm

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=phis, y=Tprim, name="primary flame T", line=dict(color=COLORWAY[4], width=3))
fig.add_scatter(x=phis, y=Texit, name="turbine-inlet T", line=dict(color=COLORWAY[0], width=3, dash="dash"))
fig.add_scatter(x=phis, y=NOprim, name="primary NO", line=dict(color="black", width=3), secondary_y=True)
fig.update_xaxes(title_text="primary equivalence ratio  phi_primary")
fig.update_yaxes(title_text="temperature [K]", secondary_y=False)
fig.update_yaxes(title_text="primary-zone NO [ppm]", secondary_y=True)
fig.update_layout(title="RQL air-split sweep (fixed overall phi = 0.5)")
fig

## Same answer as the hand-wired hard closure

The sticky marker automates exactly what a user would otherwise wire by hand -- frozen approach edges, equilibrium everywhere from the flame onward (rich *and* lean).
The two give the same mean flow.

In [ ]:
HARD = [EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL, EQ_KERNEL, EQ_KERNEL]
sol_hard = build_rql(phi_primary=1.6, edge_models=HARD).solve()  # same operating point as `net`
print("hard-closure converged:", sol_hard.converged)
print("max |dT| auto marker vs hand-wired hard closure:",
      float(np.max(np.abs(sol.field("T") - sol_hard.field("T")))), "K")

## Export for Nemo

`net` (a `Network`) and `sol` (a `Solution`) are the top-level objects the UI bridge reads.
Save either to a UI-readable YAML -- `sol.to_yaml(path)` embeds the mean-flow result fields, `net.to_yaml(path)` writes the topology only -- then open the file in Nemo.

In [ ]:
import os, tempfile

_out = os.path.join(tempfile.mkdtemp(), "rql_combustor.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; net.to_yaml(_out) for topology only
print("saved case:", _out)